# exp144_learned_likelihood_hidden_stress_and_rawtest_parity train

exp127 learned likelihood add-only feature family の hidden-like stress と raw-test/full-train parity を監査する。新規学習や submission 生成は行わない。


## Contents

1. Setup and configuration
2. Input inventory
3. Hidden-like stress and parity audit
4. Metrics and artifacts


## 1. Setup and configuration


In [ ]:
from pathlib import Path
import json
import pandas as pd

from settings import ExperimentPaths, allow_local_notebook_execution, get_nested, is_kaggle_runtime, load_config
from learned_likelihood_hidden_stress_and_rawtest_parity import main

if not is_kaggle_runtime() and not allow_local_notebook_execution():
    raise RuntimeError("Kaggle Notebook execution is canonical. Set EXPERIMENT_ALLOW_LOCAL=1 only for explicit local smoke debugging.")

config = load_config()
paths = ExperimentPaths()
print("experiment:", get_nested(config, "experiment.name"))
print("route:", get_nested(config, "experiment.route"))
print("mode:", get_nested(config, "audit.mode"))
print("parent:", get_nested(config, "lineage.parent"))
print("split parent:", get_nested(config, "lineage.split_parent"))
print("feature parent:", get_nested(config, "lineage.feature_parent"))


## 2. Input inventory


In [ ]:
input_keys = [
    "data.exp127_predictions",
    "data.exp127_feature_schema",
    "data.exp127_summary",
    "data.exp112_ml_features",
    "data.exp112_feature_schema",
    "data.exp112_summary",
    "data.exp115_fold_assignments",
    "data.exp115_well_metadata",
]
for key in input_keys:
    value = get_nested(config, key)
    path = paths.root / value if isinstance(value, str) and not Path(value).is_absolute() else Path(value)
    print(key, "exists=" + str(path.exists()), path)


In [ ]:
pred_path = paths.root / get_nested(config, "data.exp127_predictions")
schema_path = paths.root / get_nested(config, "data.exp127_feature_schema")
split_path = paths.root / get_nested(config, "data.exp115_fold_assignments")
for label, path in [("exp127 predictions", pred_path), ("exp127 schema", schema_path), ("exp115 folds", split_path)]:
    if path.exists():
        preview = pd.read_csv(path, nrows=3)
        print("\n", label, path)
        display(preview)
    else:
        print("missing locally; expected via Kaggle input:", label, path.name)


## 3. Hidden-like stress and parity audit


In [ ]:
summary = main()
print(json.dumps(summary["decision"], indent=2, ensure_ascii=False))


## 4. Metrics and artifacts


In [ ]:
artifact_dir = paths.artifacts_dir
print("artifact_dir:", artifact_dir)
for path in sorted(artifact_dir.glob("exp144_learned_likelihood_hidden_stress_and_rawtest_parity_*")):
    print(path.name, path.stat().st_size)

summary_path = artifact_dir / "exp144_learned_likelihood_hidden_stress_and_rawtest_parity_summary.json"
if summary_path.exists():
    loaded = json.loads(summary_path.read_text())
    display(pd.DataFrame(loaded.get("lgb_mean_delta_focus", [])))

checklist_path = artifact_dir / "exp144_learned_likelihood_hidden_stress_and_rawtest_parity_rawtest_parity_checklist.csv"
if checklist_path.exists():
    display(pd.read_csv(checklist_path))
